# Experimento — Oscar

**Hackathon MIAX 2026 · Predicción autorregresiva de 6 índices financieros**

- **Problema**: predecir 252 días **día a día** (rollout autorregresivo — la predicción de cada día alimenta el siguiente; los errores se acumulan).
- **Métrica**: **RMSE medio sobre los 6 índices**. Penaliza al cuadrado → los volátiles (**A** alta vol., **F** extrema/cripto) dominan el error.
- **Estrategia**: **un enfoque independiente por índice** (baseline plano, NN, o derivado de otro índice).

**Flujo**: celdas 1-4 (baselines = el *aprobado*, RMSE<75000) → pistas → NN donde aporte → **backtest 252d SIEMPRE antes de guardar** → exportar mejor enfoque por índice.

> ⚠️ El **backtest autorregresivo a 252d** es el único juez fiable (no el loss de entrenamiento). Solo **6 entregas** — no quemar intentos.

In [ ]:
# === 1 · GPU workaround — SIEMPRE primero, ANTES de importar TF/Keras ===
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"   # RTX 5070 Ti (Blackwell) incompatible con TF-GPU

import sys; sys.path.insert(0, os.getcwd())
import json
import numpy as np
import pandas as pd
import utils                       # toda la fontanería vive aquí

print("OK — CPU forzada. utils importado.")

In [ ]:
# === 2 · Config compartida — NO tocar las constantes individualmente ===
# Se acuerdan UNA VEZ entre los 3 y nadie las cambia en su notebook: cambiarlas
# invalida la comparabilidad de los backtests entre miembros (split distinto).
print("V_IN_SHARED :", utils.V_IN_SHARED, " <-- PROVISIONAL: reacordar al ver los datos reales")
print("                 (si las series son cortas, 20 puede ser excesivo para rollout 252d)")
print("VAL_DAYS    :", utils.VAL_DAYS, " (validación interna = horizonte de producción)")
print("N_DAYS_PRED :", utils.N_DAYS_PRED)
print("INDEX_COLS  :", utils.INDEX_COLS)
print("RANDOM_SEED :", utils.RANDOM_SEED)

# Loss de entrenamiento — lo ÚNICO que cada uno varía libremente aquí.
# 'mse' optimiza RMSE directamente (persigue los picos de A/F, bueno para RMSE pero
#       puede divergir más en el rollout); 'mae' es más estable ante esos picos.
# La decisión la toma el RMSE del BACKTEST autorregresivo, NO la teoría.
LOSS = 'mse'   # <-- cambiar a 'mae' para comparar estabilidad en el rollout a 252d

# Red de seguridad anti-divergencia del rollout. 0.5 = movimiento diario maximo ~+-65%
# (exp(0.5)~=1.65). Ningun indice financiero realista lo supera a diario. Si el
# diagnostico (celda 3) revela que algun indice (probablemente Index_F/cripto) tiene
# movimientos diarios reales por encima de eso, subir el clip SOLO para ese indice
# (en su llamada a backtest_autoregressive) o se recortaran movimientos legitimos.
CLIP_LOGRET = 0.5   # se propaga a backtest_autoregressive en las celdas 6 y 7

In [ ]:
# === 3 · Carga + diagnóstico (sin asumir nada sobre escala, signo ni magnitud) ===
data = utils.load_hackathon_data()        # robusto a ficheros ausentes (avisa, no falla)

idx = data.get('train_indices')
print("\nAuxiliares disponibles:", [k for k in data if k != 'train_indices'])
if idx is not None:
    print("train_indices:", idx.shape, "|", idx.index.min(), "->", idx.index.max())
    print("\nNaN por columna:"); print(idx.isnull().sum().to_string())
    print("\nEscala de cada índice (NO asumir nada — el umbral 75000 no informa de magnitud):")
    display(idx.describe().T[['mean', 'std', 'min', 'max']])

In [ ]:
# === 4 · Baselines PRIMERO — saber qué batir y asegurar el aprobado (RMSE<75000) ===
# Baselines y NN se miden en la MISMA escala (precios reconstruidos vía backtest 252d),
# por lo que son comparables 1:1. El mejor baseline por índice es el mínimo a superar.
bl = utils.eval_all_baselines(data)       # backtest de flat / drift / random_walk
display(bl)
print("Mejor baseline POR ÍNDICE (mínimo a batir con NN):")
print(bl.drop(columns='mean_rmse').min().round(1).to_string())

In [ ]:
# === 5 · Detective de pistas (opcional) =====================================
# Ghost (Index_D): sigue con LAG una señal oculta de OTRO índice. Buscar lag de corr máx.
rets = np.log(idx).diff().dropna()                       # log-rets de los 6 índices
corr_ghost = utils.lagged_correlation(rets, 'Index_D', max_lag=30)
peak = corr_ghost.abs().stack().idxmax()                 # (lag, columna) de |corr| máx
print(f"Ghost  ->  Index_D ~ {peak[1]}  con lag {peak[0]}d   (corr = {corr_ghost.loc[peak]:.3f})")
display(corr_ghost.abs().max().sort_values(ascending=False).rename('max|corr|').head())

# Huecos para las otras pistas (descomentar cuando estén los datos auxiliares):
#   Index_C ↔ macro (oro/crudo/tipos):
#     macro = utils.align_aux_features(idx, data['train_macro'], data['train_macro'].columns)
#   Index_F ↔ network (on-chain):
#     net   = utils.align_aux_features(idx, data['train_network'], data['train_network'].columns)

In [ ]:
# === 6 · Un modelo por índice (NO barrido — una arquitectura, backtest a 252d) ===
# Probar otra arquitectura = cambiar ARQ a 'dense'/'cnn1d'/'cnn_lstm'. Hacerlo SOLO si
# el backtest lo justifica (RMSE de rollout, no el loss de entrenamiento).
# dir = accuracy direccional: señal SECUNDARIA, NO fiable aislada (un modelo que prediga
#       log-ret constante >0 acierta dirección casi siempre con RMSE pésimo). Juzgar por RMSE.
ARQ, EPOCHS, V_IN = 'lstm', 100, utils.V_IN_SHARED       # lstm = natural para rollout secuencial

resultados = {}                                          # {índice: {config + rmse backtest}}
for col in utils.INDEX_COLS:
    if col not in idx.columns:
        continue
    serie = idx[col].dropna().values
    X, y = utils.make_window_dataset(serie[:-utils.VAL_DAYS], V_IN, use_log_rets=True)
    cut = int(len(X) * 0.85)                              # split interno solo para callbacks
    model = utils.build_model(ARQ, V_IN, n_features=1, n_targets=1, loss=LOSS)
    utils.train_model(model, X[:cut], y[:cut], X[cut:], y[cut:], epochs=EPOCHS)
    predict_fn = lambda x, m=model: m.predict(x, verbose=0).ravel()[0]   # m=model: evita closure-bug
    bt = utils.backtest_autoregressive(predict_fn, serie, log_ret_mode=True, clip_logret=CLIP_LOGRET)
    resultados[col] = {'arq': ARQ, 'loss': LOSS, 'v_in': V_IN, 'epochs': EPOCHS,
                       'rmse': bt['rmse'], 'dir': bt['dir_accuracy'], '_model': model}
    print(f"{col}:  RMSE rollout = {bt['rmse']:9.1f}   dir = {bt['dir_accuracy']:.2f}")

In [ ]:
# === 7 · Ensemble de semillas (palanca anti-varianza) — donde el backtest lo justifique ===
# Promediar 5 seeds elimina el ruido de inicialización. Invertir el esfuerzo en los índices
# que más pesan en el RMSE (A y F) o donde un solo modelo dé RMSE inestable.
ENSEMBLE_INDICES = ['Index_A', 'Index_F']                # <-- ajustar según tabla de la celda 6
for col in ENSEMBLE_INDICES:
    if col not in idx.columns:
        continue
    serie = idx[col].dropna().values
    X, y = utils.make_window_dataset(serie[:-utils.VAL_DAYS], V_IN, use_log_rets=True)
    cut = int(len(X) * 0.85)
    ens = utils.train_ensemble(ARQ, X[:cut], y[:cut], X[cut:], y[cut:],
                               v_in=V_IN, n_seeds=5, epochs=EPOCHS, loss=LOSS)
    bt = utils.backtest_autoregressive(ens['predict_fn'], serie, log_ret_mode=True, clip_logret=CLIP_LOGRET)
    print(f"{col}:  ensemble RMSE = {bt['rmse']:.1f}   (single = {resultados[col]['rmse']:.1f})")
    if bt['rmse'] < resultados[col]['rmse']:              # solo reemplaza si MEJORA el backtest
        resultados[col] = {'arq': ARQ, 'loss': LOSS, 'v_in': V_IN, 'epochs': EPOCHS,
                           'rmse': bt['rmse'], 'dir': bt['dir_accuracy'],
                           '_model': ens['models'], '_ensemble': True}
        print(f"   -> ensemble mejora: reemplaza el modelo de {col}")

In [ ]:
# === 8 · Guardar mejor enfoque POR ÍNDICE → results/oscar.json + models/ ===
OWNER = "oscar"
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

config_export = {}
for col, r in resultados.items():
    config_export[col] = {k: v for k, v in r.items() if not k.startswith('_')}
    m = r.get('_model')
    if r.get('_ensemble'):                                # lista de modelos (ensemble)
        files = []
        for i, sub in enumerate(m):
            fn = f'{OWNER}_{col}_seed{i}.keras'; sub.save(f'models/{fn}'); files.append(fn)
        config_export[col]['model_files'] = files         # varios = COMPETICION promedia
    elif m is not None:
        fn = f'{OWNER}_{col}.keras'; m.save(f'models/{fn}')
        config_export[col]['model_files'] = [fn]

rmse_medio = float(np.mean([c['rmse'] for c in config_export.values()])) if config_export else float('nan')
with open(f'results/{OWNER}.json', 'w') as f:
    json.dump({'owner': OWNER, 'rmse_mean_estimado': rmse_medio, 'per_index': config_export}, f, indent=2)

print(f"Guardado results/{OWNER}.json  +  modelos en models/")
for col, c in config_export.items():
    print(f"  {col}: RMSE backtest = {c['rmse']:.1f}")
print(f"\nRMSE MEDIO estimado (backtest 252d): {rmse_medio:.1f}   "
      f"{'[APROBADO]' if rmse_medio < 75000 else '[revisar]'}")

## 9 · Espacio libre para experimentos

Probar aquí abajo sin tocar nada de arriba. Ideas según las pistas del enunciado:

- **Otra arquitectura**: `ARQ = 'cnn_lstm'` y reentrenar un índice concreto.
- **Features auxiliares** (Index_C con macro, Index_F con network):
  `aux = utils.align_aux_features(idx, data['train_macro'], COLS)` →
  `utils.make_window_dataset(serie, V_IN, use_log_rets=True, aux_features=aux.values)`
  (el rollout necesita el `aux_data` del futuro: `test_macro` / `test_network`).
- **Ghost (Index_D)**: derivarlo directamente del índice + lag hallado en la celda 5 (puede batir a cualquier NN).
- **`LOSS = 'mae'`** y comparar estabilidad del rollout.

**Regla**: medir SIEMPRE con `utils.backtest_autoregressive(...)` y comparar contra
`resultados[col]['rmse']` antes de dar algo por bueno. El backtest es el único juez.

In [ ]:
# espacio libre
